In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import xgi
import os
import json
import ast
from preprocessing import create_protein_dataset, create_condensate_dataset, create_hypergraph, extract_protein_features, extract_condensate_features

# Proteins

## Proteins JSON Preprocessing Summary 

- Convert JSON to DataFrame
- Extract selected features (`uniprot_id`, `uniprotkb_id`, `protein_name`, `species_taxon_id`, 
        `species_name`, `gene_name`, `ensembl_id`, `iupred_score`, 
        `synthetic_condensate_count`)
- Some columns contain `NaN` values as 'n/a', ' ' (empty string), or '{}', so I converted them to `np.nan`
- `iupred_score` entries are lists but are formatted as strings, so i converted them to lists
- All rows that have a `NaN` entry for `protein_name` contain no information (except for `uniprot_id`)
- Dropped the `ensembl_id` column (more than 4k `NaN`) and then all rows that contained at least a `NaN` value (near 200 rows)

In [2]:
proteins_js = pd.read_json('CD-Code Dump 17_12_25/01/proteins.json')
proteins_js.loc[0]['protein']

{'uniprot_id': '0A0D9RHW7',
 'name': 'n/a',
 'species_taxon_id': 0,
 'species_name': 'n/a',
 'sequence': None,
 'uniprotkb_id': '',
 'gene_name': None,
 'ensembl_id': None,
 'ensembl_gene_id': None,
 'iupred_score': '{}',
 'synthetic_condensate_count': 0,
 'function': None,
 'antibody_link': None,
 'search': "'0a0d9rhw7':1 'n/a':2,3"}

In [3]:
df_proteins_js = extract_protein_features('CD-Code Dump 17_12_25/01/proteins.json', species='Homo sapiens')
df_proteins_js.shape

(4424, 9)

In [4]:
df_proteins_js.head()

,uniprot_id,uniprotkb_id,protein_name,species_taxon_id,species_name,gene_name,ensembl_id,iupred_score,synthetic_condensate_count
1,A0A024R3V0,A0A024R3V0_HUMAN,"DNA polymerase-transactivated protein 6, isofo...",9606,Homo sapiens,DNAPTP6,NaN,"[0.517562, 0.553315, 0.585406, 0.486429, 0.405...",0
3,A0A024RB53,A0A024RB53_HUMAN,Helix-destabilizing protein,9606,Homo sapiens,HNRPA1,NaN,"[0.926919, 0.932927, 0.852992, 0.849326, 0.837...",2
8,A0A087WVQ6,A0A087WVQ6_HUMAN,Clathrin heavy chain,9606,Homo sapiens,CLTC,ENST00000580081.2,"[0.139895, 0.196879, 0.125101, 0.17593, 0.1137...",0
11,A0A087WWU8,A0A087WWU8_HUMAN,Tropomyosin 3,9606,Homo sapiens,TPM3,ENST00000611659.5,"[0.414856, 0.414856, 0.42561, 0.349426, 0.3807...",0
12,A0A087WXK2,A0A087WXK2_HUMAN,Paternally expressed 10,9606,Homo sapiens,PEG10,ENST00000617526.4,"[0.868118, 0.891961, 0.808535, 0.827927, 0.846...",0


In [5]:
df_proteins_js['species_name'].unique()

<StringArray>
['Homo sapiens']
Length: 1, dtype: str

In [6]:
df_proteins_js.isna().sum()

uniprot_id                     0
uniprotkb_id                   0
protein_name                   0
species_taxon_id               0
species_name                   0
gene_name                     21
ensembl_id                    41
iupred_score                   0
synthetic_condensate_count     0
dtype: int64

In [7]:
sub = df_proteins_js[df_proteins_js['protein_name'].isna()]

In [8]:
sub.drop(labels=['uniprot_id', 'synthetic_condensate_count'], axis = 1, inplace=True)

In [9]:
np.all(sub.isna())

np.True_

In [10]:
df_proteins_js = df_proteins_js.drop(labels = ['ensembl_id'], axis=1)
df_proteins_js.isna().sum()

uniprot_id                     0
uniprotkb_id                   0
protein_name                   0
species_taxon_id               0
species_name                   0
gene_name                     21
iupred_score                   0
synthetic_condensate_count     0
dtype: int64

In [11]:
print(f"Rows: {df_proteins_js.shape[0]}, Columns: {df_proteins_js.shape[1]}")
df_proteins_js = df_proteins_js.dropna()
print(f"Rows (after filtering): {df_proteins_js.shape[0]}, Columns: {df_proteins_js.shape[1]}")

Rows: 4424, Columns: 8
Rows (after filtering): 4403, Columns: 8


## Proteins CSV Preprocessing Summary 

- Renamed columns to match the ones in the previously defined DataFrame (from the JSON file)
- Merged the two DataFrames

In [12]:
proteins = pd.read_csv('CD-Code Dump 17_12_25/01/proteins.csv')
print(f"Rows: {proteins.shape[0]}, Columns: {proteins.shape[1]}")
proteins = proteins.rename(columns={'Uniprot ID':'uniprot_id', 'Gene Name': 'gene_name', 'Name': 'protein_name', 
                                    'Biomolecular condensate count': 'biomolecular_condensate_count', 
                                    'Synthetic condensate count': 'synthetic_condensate_count', 'Species': 'species_name'})

proteins = proteins[proteins['species_name'] == 'Homo sapiens']
print(f"Rows (after filtering for species): {proteins.shape[0]}, Columns: {proteins.shape[1]}")
proteins.head()

Rows: 10616, Columns: 6
Rows (after filtering for species): 4424, Columns: 6


,uniprot_id,gene_name,protein_name,biomolecular_condensate_count,synthetic_condensate_count,species_name
2,P61604,HSPE1,"10 kDa heat shock protein, mitochondrial",1,0,Homo sapiens
6,Q15029,EFTUD2,116 kDa U5 small nuclear ribonucleoprotein com...,3,0,Homo sapiens
18,P31946,YWHAB,14-3-3 protein beta/alpha,2,0,Homo sapiens
19,P62258,YWHAE,14-3-3 protein epsilon,4,0,Homo sapiens
22,Q04917,YWHAH,14-3-3 protein eta,2,0,Homo sapiens


In [13]:
proteins.isna().sum()

uniprot_id                        0
gene_name                        21
protein_name                      0
biomolecular_condensate_count     0
synthetic_condensate_count        0
species_name                      0
dtype: int64

In [14]:
full_proteins = pd.merge(df_proteins_js, proteins[['uniprot_id', 'biomolecular_condensate_count']], on='uniprot_id', how='left')
full_proteins.head()

,uniprot_id,uniprotkb_id,protein_name,species_taxon_id,species_name,gene_name,iupred_score,synthetic_condensate_count,biomolecular_condensate_count
0,A0A024R3V0,A0A024R3V0_HUMAN,"DNA polymerase-transactivated protein 6, isofo...",9606,Homo sapiens,DNAPTP6,"[0.517562, 0.553315, 0.585406, 0.486429, 0.405...",0,1
1,A0A024RB53,A0A024RB53_HUMAN,Helix-destabilizing protein,9606,Homo sapiens,HNRPA1,"[0.926919, 0.932927, 0.852992, 0.849326, 0.837...",2,0
2,A0A087WVQ6,A0A087WVQ6_HUMAN,Clathrin heavy chain,9606,Homo sapiens,CLTC,"[0.139895, 0.196879, 0.125101, 0.17593, 0.1137...",0,1
3,A0A087WWU8,A0A087WWU8_HUMAN,Tropomyosin 3,9606,Homo sapiens,TPM3,"[0.414856, 0.414856, 0.42561, 0.349426, 0.3807...",0,1
4,A0A087WXK2,A0A087WXK2_HUMAN,Paternally expressed 10,9606,Homo sapiens,PEG10,"[0.868118, 0.891961, 0.808535, 0.827927, 0.846...",0,1


In [15]:
full_proteins.shape

(4403, 9)

In [16]:
full_proteins.dtypes

uniprot_id                          str
uniprotkb_id                        str
protein_name                        str
species_taxon_id                  int64
species_name                        str
gene_name                           str
iupred_score                     object
synthetic_condensate_count        int64
biomolecular_condensate_count     int64
dtype: object

In [17]:
full_proteins['species_taxon_id'] = full_proteins['species_taxon_id'].astype('int64')

In [18]:
full_proteins.dtypes

uniprot_id                          str
uniprotkb_id                        str
protein_name                        str
species_taxon_id                  int64
species_name                        str
gene_name                           str
iupred_score                     object
synthetic_condensate_count        int64
biomolecular_condensate_count     int64
dtype: object

In [19]:
full_proteins.isna().sum()

uniprot_id                       0
uniprotkb_id                     0
protein_name                     0
species_taxon_id                 0
species_name                     0
gene_name                        0
iupred_score                     0
synthetic_condensate_count       0
biomolecular_condensate_count    0
dtype: int64

In [ ]:
# Save file to avoid recomputation
#full_proteins.to_csv('data/protein_df.csv', index=False)

In [20]:
proteins = pd.read_csv('data/protein_df.csv')
print(proteins.shape)
proteins.head()

(4403, 9)


,uniprot_id,uniprotkb_id,protein_name,species_taxon_id,species_name,gene_name,iupred_score,synthetic_condensate_count,biomolecular_condensate_count
0,A0A024R3V0,A0A024R3V0_HUMAN,"DNA polymerase-transactivated protein 6, isofo...",9606,Homo sapiens,DNAPTP6,"[0.517562, 0.553315, 0.585406, 0.486429, 0.405...",0,1
1,A0A024RB53,A0A024RB53_HUMAN,Helix-destabilizing protein,9606,Homo sapiens,HNRPA1,"[0.926919, 0.932927, 0.852992, 0.849326, 0.837...",2,0
2,A0A087WVQ6,A0A087WVQ6_HUMAN,Clathrin heavy chain,9606,Homo sapiens,CLTC,"[0.139895, 0.196879, 0.125101, 0.17593, 0.1137...",0,1
3,A0A087WWU8,A0A087WWU8_HUMAN,Tropomyosin 3,9606,Homo sapiens,TPM3,"[0.414856, 0.414856, 0.42561, 0.349426, 0.3807...",0,1
4,A0A087WXK2,A0A087WXK2_HUMAN,Paternally expressed 10,9606,Homo sapiens,PEG10,"[0.868118, 0.891961, 0.808535, 0.827927, 0.846...",0,1


In [21]:
proteins_filtered = proteins[proteins['species_name'] == 'Homo sapiens']
proteins_filtered.shape

(4403, 9)

# Condensates

## Condensates Preprocessing Summary

- Convert JSON to DataFrame
- Extract selected features: `uid`, `condensate_name`, `species_taxon_id`, `species_name`,
        `confidence_score`, `condensate_type`
- Join with `condensates.csv` file to create a final `condensate_df.csv` file

In [22]:
condensates_js =  pd.read_json('CD-Code Dump 17_12_25/01/condensates.json')
condensates_js.loc[0]['condensate']

{'uid': '009021FA',
 'name': 'focal adhesion (FA) condensate',
 'description': 'The focal adhesion (FA) condensate is a membrane-less condensate formed by liquid-liquid phase separation (LLPS) at the FA periphery. It is improtant for focal adhesion morphology, cytoskeletal dynamics, and mechanical force sensing.',
 'species_taxon_id': 9606,
 'species_name': 'Homo sapiens',
 'confidence_score': 4,
 'condensate_type': 'biomolecular',
 'infectious_agent_species_id': None,
 'infectious_agent_species_name': None,
 'search': "'009021fa':1 'a':12 'adhesion':3,8,34 'and':38 'at':25 'by':18 'condensate':5,10,16 'cytoskeletal':36 'dynamics':37 'fa':4,9,27 'focal':2,7,33 'for':32 'force':40 'formed':17 'homo':42 'improtant':31 'is':11,30 'it':29 'less':15 'liquid':20,21 'liquid-liquid':19 'llps':24 'mechanical':39 'membrane':14 'membrane-less':13 'morphology':35 'periphery':28 'phase':22 'sapiens':43 'sensing':41 'separation':23 'the':6,26"}

In [23]:
df_condensates_js = extract_condensate_features('CD-Code Dump 17_12_25/01/condensates.json', confidence_th=None, condensate_type=None, species='Homo sapiens')
print(f"Rows: {df_condensates_js.shape[0]}, Columns: {df_condensates_js.shape[1]}")
df_condensates_js.head()

Rows: 330, Columns: 6


,uid,condensate_name,species_taxon_id,species_name,confidence_score,condensate_type
0,009021FA,focal adhesion (FA) condensate,9606,Homo sapiens,4,biomolecular
4,01ADBBD5,Synthetic Condensate 000031,9606,Homo sapiens,3,synthetic
7,02CD31EB,Synthetic Condensate 000174,9606,Homo sapiens,3,synthetic
8,02F8D883,IFN-I transcriptional condensate,9606,Homo sapiens,4,biomolecular
9,03D56D03,Tau Condensate,9606,Homo sapiens,3,biomolecular


In [24]:
df_condensates_js.isna().sum()

uid                 0
condensate_name     0
species_taxon_id    0
species_name        0
confidence_score    0
condensate_type     0
dtype: int64

In [25]:
condensates = pd.read_csv('CD-Code Dump 17_12_25/01/condensates.csv')
condensates = condensates.rename(columns={'Name':'condensate_name', 'UID':'uid', 'Condensate Type':'condensate_type',
                                          'Species Tax Id':'species_taxon_id', 'Proteins':'proteins', 'Condensatopathy':'condensatopathy', 
                                          'Confidence Score':'confidence_score'})
# Select only condensates with composed of 3 or more elements
condensates = condensates[condensates['proteins'] >= 3]
condensates.head()

,condensate_name,uid,condensate_type,species_taxon_id,proteins,DNA,RNA,C-mods,condensatopathy,confidence_score
0,53BP1 Mitotic Stress Body (MSBs),E12AB6E0,biomolecular,9606,3,Yes,No,No,No,4
5,Aggresome,3A3AD14F,biomolecular,83333,6,No,No,No,No,5
8,Anisosome,BEB5E62D,biomolecular,9606,6,No,No,Yes,No,4
9,Ankle Link Complex (ALC) Condensate,F80F25AF,biomolecular,10090,3,No,No,No,No,5
11,Arsenite-induced Necrosome,1C395EBB,biomolecular,10090,4,No,No,No,No,4


In [7]:
csv_columns = condensates[['uid', 'proteins', 'DNA', 'RNA', 'C-mods', 'condensatopathy']]
json_columns = df_condensates_js[['uid', 'condensate_name', 'species_name', 'confidence_score', 'condensate_type']]

In [8]:
condensates_full = pd.merge(json_columns, csv_columns, on='uid', how='inner')
condensates_full.head()

,uid,condensate_name,species_name,confidence_score,condensate_type,proteins,DNA,RNA,C-mods,condensatopathy
0,009021FA,focal adhesion (FA) condensate,Homo sapiens,4,biomolecular,3,No,No,No,No
1,03D56D03,Tau Condensate,Homo sapiens,3,biomolecular,5,No,Yes,Yes,Yes
2,03E7CE7B,NEDD4 condensates,Homo sapiens,5,biomolecular,3,No,No,No,No
3,08386919,Dense Fibrillar Component,Homo sapiens,4,biomolecular,8,Yes,Yes,No,No
4,936CAA69,Slit diaphragm condensate,Homo sapiens,1,biomolecular,3,No,No,No,No


In [9]:
condensates_full.shape

(107, 10)

In [ ]:
# Save to avoid recomputation
# condensates_full.to_csv('data/condensates_df.csv', index=False)

In [11]:
condensates = pd.read_csv('data/condensates_df.csv')
print(condensates.shape)
condensates.head()

(107, 10)


,uid,condensate_name,species_name,confidence_score,condensate_type,proteins,DNA,RNA,C-mods,condensatopathy
0,009021FA,focal adhesion (FA) condensate,Homo sapiens,4,biomolecular,3,No,No,No,No
1,03D56D03,Tau Condensate,Homo sapiens,3,biomolecular,5,No,Yes,Yes,Yes
2,03E7CE7B,NEDD4 condensates,Homo sapiens,5,biomolecular,3,No,No,No,No
3,08386919,Dense Fibrillar Component,Homo sapiens,4,biomolecular,8,Yes,Yes,No,No
4,936CAA69,Slit diaphragm condensate,Homo sapiens,1,biomolecular,3,No,No,No,No


## Map to Create Graph

- `condensate_id` --> `uid`
- `uniprotkb_ac` --> `uniprot_id`

In [47]:
proteins_full = pd.read_csv('data/protein_df.csv')
proteins_full[proteins_full['uniprot_id'].isin(('Q92804', 'O43791'))]

,uniprot_id,uniprotkb_id,protein_name,species_taxon_id,species_name,gene_name,iupred_score,synthetic_condensate_count,biomolecular_condensate_count
1309,O43791,SPOP_HUMAN,Speckle-type POZ protein,9606,Homo sapiens,SPOP,"[0.690604, 0.724957, 0.657645, 0.653063, 0.653...",3,3
7606,Q92804,RBP56_HUMAN,TATA-binding protein-associated factor 2N,9606,Homo sapiens,TAF15,"[0.754692, 0.76285, 0.808535, 0.827927, 0.8375...",5,7


In [ ]:
condensates_full[condensates_full['uid'] == '462A97B5']

,uid,condensate_name,species_name,confidence_score,condensate_type,proteins,DNA,RNA,C-mods,condensatopathy
225,462A97B5,Leucocyte nuclear body,Homo sapiens,0,biomolecular,21,No,No,No,No


In [18]:
mapping = pd.read_csv('CD-Code Dump 17_12_25/01/protein2cdcode_v2.1.tsv', sep='\t', header=0)
mapping.head()

,uniprotkb_ac,condensate_id,condensate_name
0,Q92804,462A97B5,Leucocyte nuclear body
1,O43791,462A97B5,Leucocyte nuclear body
2,Q14781,B57B8AD8,Heterochromatin
3,O62011,60DC040A,MUTator foci
4,O74718,89D152D3,Sup35 condensate


## Merging

In [22]:
mapping = pd.read_csv('CD-Code Dump 17_12_25/01/protein2cdcode_v2.1.tsv', sep='\t', header=0)

In [15]:
proteins = pd.read_csv('data/protein_df.csv')
condensates = pd.read_csv('data/condensates_df.csv')

In [19]:
condensates.columns

Index(['uid', 'condensate_name', 'species_name', 'confidence_score',
       'condensate_type', 'proteins', 'DNA', 'RNA', 'C-mods',
       'condensatopathy'],
      dtype='str')

In [17]:
condensates.head()

,uid,condensate_name,species_name,confidence_score,condensate_type,proteins,DNA,RNA,C-mods,condensatopathy
0,009021FA,focal adhesion (FA) condensate,Homo sapiens,4,biomolecular,3,No,No,No,No
1,03D56D03,Tau Condensate,Homo sapiens,3,biomolecular,5,No,Yes,Yes,Yes
2,03E7CE7B,NEDD4 condensates,Homo sapiens,5,biomolecular,3,No,No,No,No
3,08386919,Dense Fibrillar Component,Homo sapiens,4,biomolecular,8,Yes,Yes,No,No
4,936CAA69,Slit diaphragm condensate,Homo sapiens,1,biomolecular,3,No,No,No,No


In [20]:
condensates.describe()

,confidence_score,proteins
count,107.000000,107.000000
mean,2.542056,64.532710
std,1.487561,241.692902
min,0.000000,3.000000
25%,1.000000,3.000000
50%,3.000000,4.000000
75%,4.000000,7.500000
max,5.000000,1413.000000


In [23]:
# Merge mapping to proteins using uniprotkb_ac and uniprot_id
mapping_proteins = mapping.merge(
    proteins,
    left_on="uniprotkb_ac",
    right_on="uniprot_id",
    how="left"
)

In [24]:
mapping_proteins.shape

(13983, 12)

In [25]:
# Create incidence table of the hypergraph
inc_table = mapping_proteins.merge(
    condensates,
    left_on="condensate_id",
    right_on="uid",
    how="left"
)

In [27]:
inc_table.isna().sum()

uniprotkb_ac                        0
condensate_id                       0
condensate_name_x                   0
uniprot_id                       6724
uniprotkb_id                     6724
protein_name                     6724
species_taxon_id                 6724
species_name_x                   6724
gene_name                        6724
iupred_score                     6724
synthetic_condensate_count       6724
biomolecular_condensate_count    6724
uid                              7078
condensate_name_y                7078
species_name_y                   7078
confidence_score                 7078
condensate_type                  7078
proteins                         7078
DNA                              7078
RNA                              7078
C-mods                           7078
condensatopathy                  7078
dtype: int64

In [40]:
# Group proteins by condensate (hyperedge)
hyperedges = (
    inc_table
    .groupby("condensate_id")["uniprot_id"]
    .apply(list)
    .to_dict()
)

H = xgi.Hypergraph(hyperedges)

In [ ]:
# Set node attributes
protein_attrs = (
    proteins.set_index("uniprot_id").to_dict("index")
)

xgi.set_node_attributes(H, protein_attrs)

In [ ]:
# Set hyperedge (condensates attributes)
cond_attrs = (
    condensates
    .set_index("uid")
    .to_dict("index")
)

xgi.set_edge_attributes(H, cond_attrs)

# Test

In [2]:
condensates = create_condensate_dataset('CD-Code Dump 17_12_25/01/condensates.json', 'CD-Code Dump 17_12_25/01/condensates.csv',
                                        species='Homo sapiens', species_tax_id=9606, save_path='data/final_condensates.csv')
mapping = pd.read_csv('CD-Code Dump 17_12_25/01/protein2cdcode_v2.1.tsv', sep='\t', header=0)

proteins = create_protein_dataset('CD-Code Dump 17_12_25/01/proteins.json', 'CD-Code Dump 17_12_25/01/proteins.csv',
                                  species='Homo sapiens', species_taxon_id=9606, save_path='data/final_proteins.csv')

In [3]:
H = create_hypergraph(proteins, condensates, mapping, min_proteins=3, save_path='data/final_df.csv')